# Проверка исходных данных F1

Диагностический ноутбук этапа 1. Источник — PostgreSQL проекта, не сохранённая CSV-выборка. Запускайте в окружении проекта после `f1-pipeline collect`.

Одна строка laps — круг пилота в сессии. Итоговые модели ещё не обучались.

In [1]:
from f1_pipeline.quality import inspect_database
from f1_pipeline.db import connect
from f1_pipeline.config import ROOT
from IPython.display import display, HTML
import html

def table(rows):
    if not rows:
        return
    keys = list(rows[0])
    markup = "<table><thead><tr>" + "".join("<th>"+html.escape(str(k))+"</th>" for k in keys) + "</tr></thead><tbody>"
    for row in rows:
        markup += "<tr>" + "".join("<td>"+html.escape(format(row[k], ".3f") if isinstance(row[k], float) else str(row[k]))+"</td>" for k in keys) + "</tr>"
    display(HTML(markup+"</tbody></table>"))

report = inspect_database()

## Покрытие и пропуски

Счётчики отражают исходные данные. NULL времени круга не заменяется нулём. Число записей пилот–сессия отличается от числа уникальных людей.

In [2]:
table([{ "Год": s["year"], "Круги": s["laps"], "Отрезки": s["stints"], "Погода": s["weather"], "События": s["race_control"]} for s in report["sessions"]])
table([{ "Год": s["year"], "Нет времени круга": s["null_duration"], "Нет начала круга": s["null_start"]} for s in report["sessions"]])

Год,Круги,Отрезки,Погода,События
2023,1312,63,154,49
2024,1310,62,154,107
2025,1204,73,155,102


Год,Нет времени круга,Нет начала круга
2023,2,2
2024,1,1
2025,2,0


## Связи и погодное покрытие

Погода сопоставляется последним наблюдением с временем не позже начала круга. Круги без отрезка шин потребуют решения на этапе 2.

In [3]:
table([{ "Год": s["year"], "Нет шин": s["unmatched_stint"], "Дубли связи": s["multiple_stints"], "Нет погоды": s["missing_prior_weather"], "Задержка, с": str(s["max_weather_age_seconds"])} for s in report["join_coverage"]])
table([{ "Год": s["year"], "Мин., °C": s["minimum"], "Макс., °C": s["maximum"], "Среднее, °C": s["mean"], "σ, °C": s["stddev"]} for s in report["numeric_profiles"]["weather.track_temperature"]])

Год,Нет шин,Дубли связи,Нет погоды,"Задержка, с"
2023,0,0,0,59.999000
2024,0,0,0,59.882000
2025,2,0,0,59.948000


Год,"Мин., °C","Макс., °C","Среднее, °C","σ, °C"
2023,26.800,42.500,33.839,5.011
2024,38.300,43.600,41.096,1.059
2025,46.500,50.400,48.543,0.986


## Проверка SQL-извлечения

Целевая переменная — lap_duration. Здесь ещё присутствуют круги, которые будут исключены при подготовке. Проверяется направление временного объединения, а не готовность модели.

In [4]:
with connect() as conn:
    candidates = conn.execute((ROOT / "sql/003_extract.sql").read_text()).fetchall()
assert all(r["weather_observed_at"] is None or r["weather_observed_at"] <= r["date_start"] for r in candidates)
print("Извлечено строк:", len(candidates))
table([{ "Пилот": s["full_name"], "Круг": s["lap_number"], "Время, с": s["lap_duration"], "Шины": s["compound"], "Пробег": s["tyre_age_before_lap"]} for s in candidates[:5]])

Извлечено строк: 3826


Пилот,Круг,"Время, с",Шины,Пробег
Max VERSTAPPEN,1,84.281,MEDIUM,0
Max VERSTAPPEN,2,80.402,MEDIUM,1
Max VERSTAPPEN,3,80.499,MEDIUM,2
Max VERSTAPPEN,4,80.346,MEDIUM,3
Max VERSTAPPEN,5,80.283,MEDIUM,4


## Проверки источника и вывод

Исправления и фильтрация пока не выполнялись. Недостающие тайминги и дождевой замер требуют обработки на следующем этапе. Три гонки ограничивают независимое разнообразие погодных условий.

In [5]:
import json
assert all(b["hash_matches"] for b in report["batches"])
assert not any(report["duplicates"].values())
table([{ "severity": i["severity"], "issue": i["issue"], "details": json.dumps({k:v for k,v in i.items() if k not in ("severity", "issue")}, ensure_ascii=False, default=str)} for i in report["issues"]])

severity,issue,details
medium,"Null lap timing; preserve now, exclude or resolve in stage 2","{""year"": 2023, ""null_duration"": 2, ""null_start"": 2}"
medium,"Null lap timing; preserve now, exclude or resolve in stage 2","{""year"": 2024, ""null_duration"": 1, ""null_start"": 1}"
medium,Dry scope needs temporal rainfall filtering,"{""year"": 2024, ""rain_samples"": 1, ""rain_unknown"": 0}"
medium,"Null lap timing; preserve now, exclude or resolve in stage 2","{""year"": 2025, ""null_duration"": 2, ""null_start"": 0}"
high,Join coverage requires stage-2 treatment,"{""year"": 2025, ""laps"": 1204, ""unmatched_stint"": 2, ""multiple_stints"": 0, ""missing_prior_weather"": 0, ""max_weather_age_seconds"": ""59.948000""}"
